# Forest Fire Prediction: Data Cleaning and Initial Exploration

In this notebook, we perform the initial data loading, cleaning, and exploration of the Algerian Forest Fires dataset. A rigorous data preprocessing pipeline is crucial for building robust machine learning models.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

**What was done:** Imported essential data manipulation and visualization libraries (Pandas, NumPy, Matplotlib, Seaborn) and configured the environment (e.g., suppressing warnings, inline plotting).

**Why it was done:** These libraries form the foundational toolkit for numerical computations, data manipulation, and exploratory visualization.

**Key observations:** The environment is successfully configured and ready for data ingestion.

## 1. Data Loading

We will load the raw dataset using relative paths. Using relative paths ensures the notebook remains reproducible across different environments and operating systems.

In [3]:
# Define the relative path to the raw dataset
file_path = '../data/raw/algerian_forest_fire_dataset.csv'

# Load the dataset, skipping the first row which acts as a generic title
df = pd.read_csv(file_path, header=1)

df.head()

,day,month,year,Temperature,RH,Ws,Rain,FFMC,DMC,DC,ISI,BUI,FWI,Classes
0,1,6,2012,29,57,18,0,65.7,3.4,7.6,1.3,3.4,0.5,not fire
1,2,6,2012,29,61,13,1.3,64.4,4.1,7.6,1,3.9,0.4,not fire
2,3,6,2012,26,82,22,13.1,47.1,2.5,7.1,0.3,2.7,0.1,not fire
3,4,6,2012,25,89,13,2.5,28.6,1.3,6.9,0,1.7,0,not fire
4,5,6,2012,27,77,16,0,64.8,3,14.2,1.2,3.9,0.5,not fire


**What was done:** Loaded the raw CSV file into a Pandas DataFrame using a relative path, explicitly setting `header=1` to skip the document's main title ('Bejaia Region Dataset').

**Why it was done:** The first row in the raw CSV is not a structural column header. The actual tabular headers begin on the second row. Relative paths ensure modularity and reproducibility.

**Key observations:** The dataset contains meteorological observations (Temperature, RH, Ws, Rain) and fire weather indices (FFMC, DMC, DC, ISI, BUI, FWI) along with the target variable `Classes`.

## 2. Initial Exploration

Let's examine the basic structure, shape, and datatypes of our dataset before proceeding with targeted cleaning steps.

In [4]:
# Display basic information: shape and datatypes
print(f"Dataset Shape: {df.shape}\n")

print("Data Types and Non-Null Counts:")
df.info()

Dataset Shape: (247, 14)

Data Types and Non-Null Counts:
<class 'pandas.DataFrame'>
RangeIndex: 247 entries, 0 to 246
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   day          246 non-null    str  
 1   month        245 non-null    str  
 2   year         245 non-null    str  
 3   Temperature  245 non-null    str  
 4   RH           245 non-null    str  
 5   Ws           245 non-null    str  
 6   Rain         245 non-null    str  
 7   FFMC         245 non-null    str  
 8   DMC          245 non-null    str  
 9   DC           245 non-null    str  
 10  ISI          245 non-null    str  
 11  BUI          245 non-null    str  
 12  FWI          245 non-null    str  
 13  Classes      244 non-null    str  
dtypes: str(14)
memory usage: 37.3 KB


**What was done:** Extracted the dimensions of the dataset (rows and columns) and inspected the raw datatypes of each feature using `df.info()`.

**Why it was done:** To understand the initial state of the data, detect formatting issues, and identify non-numeric columns that should theoretically be numeric.

**Key observations:** Several numerical features (like FFMC, DMC, DC, etc.) are read as `object` (strings). This indicates the presence of non-numeric characters, extra sub-headers, or missing values within those columns.

## 3. Data Cleaning

### 3.1 Handling Multiple Regions and Extraneous Rows
The dataset sequentially lists data from two regions (Bejaia and Sidi-Bel Abbes). We will create a `Region` feature to logically separate them and remove the redundant intermediate headers and empty rows.

In [5]:
# Drop rows that are completely empty
df.dropna(how='all', inplace=True)

# Add a 'Region' column (0 for Bejaia, 1 for Sidi-Bel Abbes)
# The Bejaia region spans up to original index 122.
df.loc[:122, 'Region'] = 0
df.loc[122:, 'Region'] = 1
df['Region'] = df['Region'].astype(int)

# Remove the redundant header rows from the middle of the dataset
df = df[df['day'] != 'day']
df = df[df['day'] != 'Sidi-Bel Abbes Region Dataset']

# Reset the index after dropping rows
df.reset_index(drop=True, inplace=True)
df.head()

,day,month,year,Temperature,RH,Ws,Rain,FFMC,DMC,DC,ISI,BUI,FWI,Classes,Region
0,1,6,2012,29,57,18,0,65.7,3.4,7.6,1.3,3.4,0.5,not fire,0
1,2,6,2012,29,61,13,1.3,64.4,4.1,7.6,1,3.9,0.4,not fire,0
2,3,6,2012,26,82,22,13.1,47.1,2.5,7.1,0.3,2.7,0.1,not fire,0
3,4,6,2012,25,89,13,2.5,28.6,1.3,6.9,0,1.7,0,not fire,0
4,5,6,2012,27,77,16,0,64.8,3,14.2,1.2,3.9,0.5,not fire,0


**What was done:** Dropped fully empty rows, created an integer `Region` column (0=Bejaia, 1=Sidi-Bel Abbes), and removed the internal string header rows that separated the regions.

**Why it was done:** ML models require a consistent tabular structure. The internal headers and empty rows would corrupt our numerical features. Encoding the region preserves geographic context.

**Key observations:** The dataset is now structurally contiguous, and geographic information has been successfully transformed into a usable numerical feature.

### 3.2 Standardizing Column Names

Column names often contain hidden trailing spaces or inconsistent casing which can cause runtime errors.

In [6]:
# Strip leading and trailing spaces from all column names
df.columns = df.columns.str.strip()
print("Cleaned Columns:", df.columns.tolist())

Cleaned Columns: ['day', 'month', 'year', 'Temperature', 'RH', 'Ws', 'Rain', 'FFMC', 'DMC', 'DC', 'ISI', 'BUI', 'FWI', 'Classes', 'Region']


**What was done:** Stripped all leading and trailing whitespaces from the DataFrame column names.

**Why it was done:** To prevent frustrating `KeyError` exceptions when referencing columns (e.g., referencing `'Classes'` instead of the raw `'Classes  '`).

**Key observations:** Column names are now clean, standardized, and easily referenceable.

### 3.3 Handling Missing Values and Incorrect Datatypes

We need to check for missing values, handle corrupted rows, and convert features to their proper numeric representations.

In [7]:
# Check for missing values
print("Missing values before cleaning:\n", df.isnull().sum())

# Drop the row with missing values (corrupted data entry)
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

# Convert chronological and discrete columns to integers
int_columns = ['day', 'month', 'year', 'Temperature', 'RH', 'Ws']
for col in int_columns:
    df[col] = df[col].astype(int)

# Convert indices to floats, coercing any remaining text to NaN
float_columns = ['Rain', 'FFMC', 'DMC', 'DC', 'ISI', 'BUI', 'FWI']
for col in float_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop any newly introduced NaNs from coercion
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

print("\nData Types After Cleaning:")
print(df.info())

Missing values before cleaning:
 day            0
month          0
year           0
Temperature    0
RH             0
Ws             0
Rain           0
FFMC           0
DMC            0
DC             0
ISI            0
BUI            0
FWI            0
Classes        1
Region         0
dtype: int64

Data Types After Cleaning:
<class 'pandas.DataFrame'>
RangeIndex: 243 entries, 0 to 242
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   day          243 non-null    int64  
 1   month        243 non-null    int64  
 2   year         243 non-null    int64  
 3   Temperature  243 non-null    int64  
 4   RH           243 non-null    int64  
 5   Ws           243 non-null    int64  
 6   Rain         243 non-null    float64
 7   FFMC         243 non-null    float64
 8   DMC          243 non-null    float64
 9   DC           243 non-null    float64
 10  ISI          243 non-null    float64
 11  BUI          243 non-null 

**What was done:** Evaluated missing values, dropped a corrupted row containing missing data, and explicitly cast discrete features to `int64` and continuous features to `float64`.

**Why it was done:** Statistical modeling requires numeric arrays. Missing values and string-encoded numbers prevent algorithms from processing the data. Coercion guarantees type safety.

**Key observations:** All independent variables are now successfully converted to their correct numeric data types. The dataset is free of missing or corrupted values.

### 3.4 Cleaning the Target Variable

The target variable `Classes` contains trailing spaces and inconsistent casing. We will standardize it and convert it into a binary format for classification.

In [8]:
# Clean string inconsistencies in the Classes column
df['Classes'] = df['Classes'].astype(str).str.strip()

# Display unique values before mapping
print("Unique classes before mapping:", df['Classes'].unique())

# Convert to binary: 'fire' -> 1, 'not fire' -> 0
df['Classes'] = np.where(df['Classes'].str.contains('not fire'), 0, 1)

print("Unique classes after mapping:", df['Classes'].unique())

Unique classes before mapping: <ArrowStringArray>
['not fire', 'fire']
Length: 2, dtype: str
Unique classes after mapping: [0 1]


**What was done:** Cleaned string inconsistencies in the target variable `Classes` and mapped the categories to a binary integer format (`0` for 'not fire', `1` for 'fire').

**Why it was done:** Machine learning algorithms natively handle numerical targets for classification tasks much better than string labels. Binary encoding is essential for algorithms like Logistic Regression.

**Key observations:** The target is now perfectly binarized and ready for supervised learning algorithms.

### 3.5 Removing Duplicates

Finally, we will verify if there are any duplicated records and remove them to ensure dataset integrity.

In [9]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")

# Remove duplicates if they exist
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print("Duplicates removed successfully.")

Number of duplicate rows: 0


**What was done:** Calculated the number of exact duplicate rows and removed them from the dataset.

**Why it was done:** Duplicate data can artificially inflate model performance metrics and bias the algorithm towards repeated observations.

**Key observations:** The dataset is now fully sanitized, featuring only unique, distinct observations.

## 4. Summary Statistics

With a pristine dataset, we can securely compute summary statistics to understand the numerical distribution of our features.

In [10]:
# Display summary statistics for numerical features
df.describe().T

,count,mean,std,min,25%,50%,75%,max
day,243.0,15.761317,8.842552,1.0,8.00,16.0,23.00,31.0
month,243.0,7.502058,1.114793,6.0,7.00,8.0,8.00,9.0
year,243.0,2012.000000,0.000000,2012.0,2012.00,2012.0,2012.00,2012.0
Temperature,243.0,32.152263,3.628039,22.0,30.00,32.0,35.00,42.0
RH,243.0,62.041152,14.828160,21.0,52.50,63.0,73.50,90.0
Ws,243.0,15.493827,2.811385,6.0,14.00,15.0,17.00,29.0
Rain,243.0,0.762963,2.003207,0.0,0.00,0.0,0.50,16.8
FFMC,243.0,77.842387,14.349641,28.6,71.85,83.3,88.30,96.0
DMC,243.0,14.680658,12.393040,0.7,5.80,11.3,20.80,65.9
DC,243.0,49.430864,47.665606,6.9,12.35,33.1,69.10,220.4


**What was done:** Generated a statistical summary containing measures of central tendency (mean) and dispersion (std, min, max, quartiles) for all numeric columns.

**Why it was done:** To detect potential outliers, understand the scale of different features, and guide subsequent normalization or standardization decisions.

**Key observations:** 
- The dataset is roughly evenly split between the two regions (mean of Region ~ 0.5).
- Features like `DC` and `BUI` have significantly higher maximum values and standard deviations compared to `Temperature` or `Ws`. This strongly suggests that feature scaling (e.g., StandardScaler) will be required before training distance-based or gradient-based models.

In [11]:
# Save the cleaned dataset for downstream processes
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/algerian_forest_fire_cleaned.csv', index=False)
print("Cleaned dataset saved successfully to 'data/processed/algerian_forest_fire_cleaned.csv'")

Cleaned dataset saved successfully to 'data/processed/algerian_forest_fire_cleaned.csv'


**What was done:** Exported the final cleaned DataFrame to the `data/processed/` directory.

**Why it was done:** To cleanly decouple the data preparation stage from feature engineering and modeling. This modular approach is a standard MLOps best practice.

**Key observations:** The data cleaning pipeline is complete. The resulting dataset is structurally sound, mathematically safe, and ready for advanced EDA or model training.